# AirSense: Notebook 01 — Exploratory Data Analysis & Data Cleaning

**Project:** AirSense — Air Quality Prediction and Analysis Using Machine Learning  
**Dataset:** India Air Quality (2015–2020), `city_day.csv`  
**Objective:** Empirical investigation of pollutant distributions, missingness patterns, CPCB target class distributions, spatial-temporal variations, and leak-free preprocessing.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_city_day, POLLUTANT_COLUMNS, CPCB_AQI_CATEGORIES
from src.preprocessing import (
    filter_labeled_data,
    extract_temporal_features,
    create_stratified_split,
    fit_imputer,
    transform_impute,
    save_processed_splits
)

sns.set_theme(style='whitegrid')
print('Libraries and custom modules loaded successfully!')


## 1. Data Ingestion & Target Label Filtering
Under CPCB standards, observations missing `AQI_Bucket` cannot be reliably evaluated for classification.
In strict accordance with project rules, target labels are **never imputed**.


In [ ]:
# Load raw city_day.csv
df_raw = load_city_day(parse_dates=True, drop_na_target=False)
print('Raw dataset shape:', df_raw.shape)

# Filter observations with valid ground-truth target
df_labeled, audit = filter_labeled_data(df_raw, target_col='AQI_Bucket')
print(f"Dropped {audit['dropped_missing_target']} rows ({audit['dropped_pct']}%) with missing AQI_Bucket")
print(f"Retained {len(df_labeled)} labeled rows for EDA and modeling")
df_labeled.head(5)


## 2. Missing-Value Frequency & Feature Sparsity
Comparing missingness between the raw dataset and the labeled subset.


In [ ]:
missing_summary = pd.DataFrame({
    'Raw_Null_Count': df_raw.isnull().sum(),
    'Raw_Null_Pct': (df_raw.isnull().mean() * 100).round(2),
    'Labeled_Null_Count': df_labeled.isnull().sum(),
    'Labeled_Null_Pct': (df_labeled.isnull().mean() * 100).round(2)
})
missing_summary


## 3. Target Distribution (`AQI_Bucket`) & Severe Class Imbalance
Evaluating the prevalence of each official CPCB category.


In [ ]:
counts = df_labeled['AQI_Bucket'].value_counts().reindex(CPCB_AQI_CATEGORIES)
percentages = (counts / len(df_labeled) * 100).round(2)
cpcb_table = pd.DataFrame({'Count': counts, 'Percentage': percentages})
print(cpcb_table)

plt.figure(figsize=(9, 5))
colors = ['#2ecc71', '#a8e6cf', '#f39c12', '#e67e22', '#e74c3c', '#7b1fa2']
bars = plt.bar(counts.index, counts.values, color=colors, edgecolor='black')
plt.title('CPCB AQI Category Distribution (Severe Imbalance)', weight='bold')
plt.ylabel('Observations')
for b, c, p in zip(bars, counts.values, percentages.values):
    plt.text(b.get_x() + b.get_width()/2, c + 150, f"{c:,}\n({p}%)", ha='center', weight='bold')
plt.tight_layout()
plt.show()


## 4. Pollutant Distributions & Right-Skewness
Examining continuous distributions of criteria pollutants.


In [ ]:
cols_to_stat = [c for c in POLLUTANT_COLUMNS if c in df_labeled.columns] + ['AQI']
stats_df = df_labeled[cols_to_stat].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
stats_df['skewness'] = df_labeled[cols_to_stat].skew().round(2)
stats_df


## 5. Inter-Pollutant Correlation Heatmap
Assessing linear relationships between criteria pollutants and continuous AQI.


In [ ]:
corr = df_labeled[cols_to_stat].corr()
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', square=True, vmin=-0.2, vmax=1.0)
plt.title('Pearson Correlation Matrix of Criteria Pollutants & AQI', weight='bold')
plt.show()


## 6. City-Level Disparities (26 Cities)
Ranking cities by median AQI and proportion of Severe air days.


In [ ]:
city_summary = df_labeled.groupby('City').agg(
    Median_AQI=('AQI', 'median'),
    Severe_Days_Pct=('AQI_Bucket', lambda x: round((x == 'Severe').mean() * 100, 2)),
    Total_Days=('AQI', 'count')
).sort_values('Median_AQI', ascending=False)
city_summary


## 7. Temporal & Seasonal Dynamics
Analyzing seasonality across Winter, Summer, Monsoon, and Post-Monsoon.


In [ ]:
df_seasonal = extract_temporal_features(df_labeled)
seasonal_aqi = df_seasonal.groupby('Season')['AQI'].agg(['median', 'mean', 'count']).reindex(['Winter', 'Summer', 'Monsoon', 'Post-Monsoon'])
print(seasonal_aqi)

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_seasonal, x='Season', y='AQI', order=['Winter', 'Summer', 'Monsoon', 'Post-Monsoon'], showfliers=False)
plt.title('Seasonal AQI Variation (Winter Peak vs. Monsoon Cleansing)', weight='bold')
plt.show()


## 8. Stratified Train / Validation / Test Splitting
Preserving class distributions with zero data leakage (70% Train, 15% Val, 15% Test).


In [ ]:
train_df, val_df, test_df = create_stratified_split(df_labeled, target_col='AQI_Bucket', test_size=0.15, val_size=0.15, random_state=42)
print(f"Train split: {len(train_df):,} rows ({len(train_df)/len(df_labeled)*100:.1f}%)")
print(f"Val split:   {len(val_df):,} rows ({len(val_df)/len(df_labeled)*100:.1f}%)")
print(f"Test split:  {len(test_df):,} rows ({len(test_df)/len(df_labeled)*100:.1f}%)")

# Fit imputer on training fold ONLY
imputer = fit_imputer(train_df, feature_cols=POLLUTANT_COLUMNS, group_col='City')
train_imp = transform_impute(train_df, imputer)
val_imp = transform_impute(val_df, imputer)
test_imp = transform_impute(test_df, imputer)

# Persist to data/processed/
saved_files = save_processed_splits(train_imp, val_imp, test_imp, output_dir='data/processed')
print('Processed splits saved successfully to:', saved_files)
